# Transparency, Confidence and Bank Runs
### Interactive companion to the paper — Marco Fantin

This notebook lets you explore the model of the paper by moving a slider for each parameter.

**How to use it:** choose *Run → Run All Cells* (or *Kernel → Restart Kernel and Run All Cells*). The notebook must sit in the same folder as `model.py` and `plotting.py`.

| Section | Content |
|---|---|
| 1 | Setup |
| 2 | The model in numbers: two scenarios side by side |
| 3 | Interactive explorer |

All parameter values are illustrative, not calibrated to any specific bank.

## 1. Setup

We import the model (`model.py`), the charts (`plotting.py`) and the widgets that create the sliders.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from model import Params, summary          # the economics
from plotting import apply_style, draw_dashboard   # the charts

%matplotlib inline
apply_style()

# Baseline scenario of the paper
BASELINE = Params(tau=1.0, s=0.2, d=0.5, ell=0.30,
                  lam=0.05, cbar=-0.5, sigma=0.5)

## 2. The model in numbers

Two banks that differ **only** in the severity of the crisis. Look at the row *dE[F]/dtau*: positive in the mild crisis (transparency helps), negative in the severe one (transparency hurts).

In [2]:
scenarios = {
    "Mild crisis (s = 0.2)":   BASELINE,
    "Severe crisis (s = 1.0)": BASELINE.with_(s=1.0),
}

table = pd.DataFrame({name: summary(p) for name, p in scenarios.items()})
table

,Mild crisis (s = 0.2),Severe crisis (s = 1.0)
theta (true soundness),0.3,-0.5
omega (weight on disclosure),0.5,0.5
E[F] (expected confidence),0.15,-0.25
dE[F]/dtau (effect of transparency),0.075,-0.125
F* (critical confidence),-0.18318,-0.18318
w (share withdrawing),0.14196,0.343111
run,False,True
tau* (critical transparency),NaN,0.578183


## 3. Interactive explorer

Three panels, one for each step of the model:

- **(A) Confidence** — how expected confidence changes with transparency, against the run threshold $F^*$.
- **(B) Withdrawals** — how many depositors leave as confidence falls (read right to left), against the liquidity buffer $\ell$.
- **(C) Run map** — every combination of transparency and crisis severity: red = run, blue = no run. The dot is the current bank.


In [3]:
# -----------------------------------------------------------------------------
# 3.1  The sliders: one per parameter
# -----------------------------------------------------------------------------
def make_slider(name, label, low, high, step):
    """A slider that starts at the baseline value of parameter `name`."""
    return widgets.FloatSlider(
        value=getattr(BASELINE, name), min=low, max=high, step=step,
        description=label,
        continuous_update=False,          # redraw only when released
        readout_format=".2f",
        style={"description_width": "160px"},
        layout=widgets.Layout(width="450px"),
    )

sliders = {
    # The bank and the crisis
    "tau":   make_slider("tau",   "Transparency τ",        0.00, 6.00, 0.05),
    "s":     make_slider("s",     "Crisis severity s",     0.00, 2.00, 0.01),
    "d":     make_slider("d",     "Pre-crisis margin d",  -1.00, 1.00, 0.01),
    "ell":   make_slider("ell",   "Liquidity buffer ℓ",    0.05, 0.95, 0.01),
    # The depositors
    "lam":   make_slider("lam",   "Non-rational share λ",  0.00, 0.50, 0.01),
    "cbar":  make_slider("cbar",  "Avg. threshold c̄",     -1.50, 0.50, 0.01),
    "sigma": make_slider("sigma", "Heterogeneity σ",       0.05, 1.50, 0.01),
}

# Reset button: puts every slider back to its baseline value
reset_button = widgets.Button(description="Reset")

def reset(_):
    for name, slider in sliders.items():
        slider.value = getattr(BASELINE, name)

reset_button.on_click(reset)

In [4]:
# -----------------------------------------------------------------------------
# 3.2  What happens when a slider moves: redraw the three panels
# -----------------------------------------------------------------------------
def show_model(tau, s, d, ell, lam, cbar, sigma):
    """Build the scenario from the sliders and draw the dashboard."""
    scenario = Params(tau=tau, s=s, d=d, ell=ell,
                      lam=lam, cbar=cbar, sigma=sigma)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    text, colour = draw_dashboard(axes, scenario)
    fig.suptitle(text, color=colour, fontweight="bold", fontsize=11, y=1.06)
    fig.tight_layout()
    plt.show()


# -----------------------------------------------------------------------------
# 3.3  Layout: sliders in two columns, charts below
# -----------------------------------------------------------------------------
left_column = widgets.VBox([sliders[k] for k in ("tau", "s", "d", "ell")])
right_column = widgets.VBox([sliders[k] for k in ("lam", "cbar", "sigma")]
                            + [reset_button])
charts = widgets.interactive_output(show_model, sliders)

display(widgets.HBox([left_column, right_column]), charts)

Output()